
# 03 — Preprocessing (Unified Pipeline)

**Purpose.** Prepare a clean, auditable, and site‑aware dataset for downstream **Feature Engineering (04_FE)**.  
This notebook implements a *pipeline-in-a-notebook*: clear, reviewable steps (cells) with functionized code.

**Key design rules**
- **UTC everywhere.** All timestamps are timezone‑aware in UTC (as provided by the project).
- **Dynamic configuration.** Paths, site selection, and behaviors are loaded from YAML files:
  - `project_config.SITES.yaml` — per‑site directory structure, exports, and toggles
  - `project_config.EDA.yaml` — EDA‑stage options (referenced for consistency)
  - `decisions_preprocess.yaml` — explicit cleaning decisions and thresholds
- **Auditability.** Every change is summarized into a change log and exported as reports.
- **Conservatism.** We avoid aggressive "repairs". No anomaly modeling yet; this comes right after this notebook.

> **Outputs.** A single, base‑clean dataset (no resampling, no anomaly repairs), plus reports.



## Table of Contents
1. [Imports & Global Settings](#imports)
2. [Config Loading (Dynamic Paths)](#config)
3. [Load Inputs](#load)
4. [Schema Harmonization & DType Enforcement](#schema)
5. [Time Index QA (UTC / continuity)](#timeqa)
6. [Base Cleaning (Guided by `decisions_preprocess.yaml`)](#clean)
7. [Missing Data Summary (Before/After)](#missing)
8. [Value Ranges, Types & Stats Report](#stats)
9. [Integrity & Diff Reports](#reports)
10. [Exports & Hand‑off to 04_FE](#exports)



<a id="imports"></a>

## 1) Imports & Global Settings

**Rationale.** Centralize imports and basic settings (random seeds, display). We use only standard, reproducible libs.


In [1]:

# --- Cell 1: Imports & Settings ---
from __future__ import annotations

import os
import sys
import json
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import yaml

# Visualization (used only for summaries/QA if needed)
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

TARGET_COL = 'actual_power_kW' 

print("[ENV] pandas", pd.__version__)
print("[ENV] numpy", np.__version__)


[ENV] pandas 2.3.3
[ENV] numpy 2.2.5



<a id="config"></a>

## 2) Config Loading (Dynamic Paths)

**Rationale.** Multi-site, dynamic paths. We resolve YAMLs from (1) current working dir, (2) `PROJECT_CONFIG_DIR` env var, (3) `/mnt/data` fallback.  
This fixes cases where the notebook runtime isn't launched from the project root.


In [2]:
# --- Cell 2: Load unified YAML configs and resolve base paths ---
import yaml
import json
from pathlib import Path

# === 1. Define explicit base paths ===
PROJECT_ROOT = Path(r"C:\Users\baraky\PycharmProjects\solar_ds_capstone")
CONFIGS_DIR  = PROJECT_ROOT / "configs"

# === 2. Load YAMLs directly from configs folder ===
SITES_YAML      = CONFIGS_DIR / "project_config.SITES.yaml"
EDA_YAML        = CONFIGS_DIR / "project_config.EDA.yaml"
DECISIONS_YAML  = CONFIGS_DIR / "decisions_preprocess.yaml"

def read_yaml(path: Path) -> dict:
    """Safe YAML loader with helpful error if missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing YAML file: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

sites_cfg  = read_yaml(SITES_YAML)
eda_cfg    = read_yaml(EDA_YAML)
decisions  = read_yaml(DECISIONS_YAML)

# === 3. Extract key paths from project_config.SITES.yaml ===
paths = sites_cfg.get("paths", {})
RAW_BASE_DIR       = Path(paths.get("raw_base_dir"))
MERGED_SITE_DIR    = Path(paths.get("merged_site_dir"))
SITE_PARQUET_NAME  = paths.get("site_parquet_name")
MERGED_BASE_PARQUET= Path(paths.get("merged_base_parquet"))

# === 4. Load complementary parameters from the other YAMLs ===
eda_root       = eda_cfg.get("eda", {})
EDA_RESOLUTION = eda_root.get("resolution", "15min")
EDA_OUTPUT_DIR = Path(eda_root.get("output_dir", PROJECT_ROOT / "outputs" / "EDA"))

selection_sites = sites_cfg.get("selection", {}).get("sites", ["site1"])
poc_cfg         = sites_cfg.get("poc", {})
POC_ENABLE      = bool(poc_cfg.get("enable", False))
POC_SITE        = poc_cfg.get("site_id") or selection_sites[0]

# === 5. Combine into one configuration dictionary ===
cfg = {
    "RAW_BASE_DIR": str(RAW_BASE_DIR),
    "MERGED_SITE_DIR": str(MERGED_SITE_DIR),
    "SITE_PARQUET_NAME": SITE_PARQUET_NAME,
    "MERGED_BASE_PARQUET": str(MERGED_BASE_PARQUET),
    "SELECTION_SITES": selection_sites,
    "POC": {"enable": POC_ENABLE, "site_id": POC_SITE},
    "EDA": {
        "resolution": EDA_RESOLUTION,
        "output_dir": str(EDA_OUTPUT_DIR),
    },
}

# === 6. Display loaded configuration ===
print("[CONFIG] Loaded YAMLs:")
print("  SITES_YAML     =", SITES_YAML)
print("  EDA_YAML       =", EDA_YAML)
print("  DECISIONS_YAML =", DECISIONS_YAML)

print("\n[CONFIG] Effective configuration:")
print(json.dumps(cfg, indent=2, ensure_ascii=False))

print("\n[DECISIONS] Top-level keys:", list(decisions.keys()))


[CONFIG] Loaded YAMLs:
  SITES_YAML     = C:\Users\baraky\PycharmProjects\solar_ds_capstone\configs\project_config.SITES.yaml
  EDA_YAML       = C:\Users\baraky\PycharmProjects\solar_ds_capstone\configs\project_config.EDA.yaml
  DECISIONS_YAML = C:\Users\baraky\PycharmProjects\solar_ds_capstone\configs\decisions_preprocess.yaml

[CONFIG] Effective configuration:
{
  "RAW_BASE_DIR": "C:\\Users\\baraky\\PycharmProjects\\solar_ds_capstone\\data\\raw\\Ktora",
  "MERGED_SITE_DIR": "C:\\Users\\baraky\\PycharmProjects\\solar_ds_capstone\\data\\processed\\Ktora",
  "SITE_PARQUET_NAME": null,
  "MERGED_BASE_PARQUET": "C:\\Users\\baraky\\PycharmProjects\\solar_ds_capstone\\data\\processed\\Ktora",
  "SELECTION_SITES": [
    "site2"
  ],
  "POC": {
    "enable": true,
    "site_id": "site1"
  },
  "EDA": {
    "resolution": "1h",
    "output_dir": "C:\\Users\\baraky\\PycharmProjects\\solar_ds_capstone\\outputs\\EDA"
  }
}

[DECISIONS] Top-level keys: ['version', 'generated_utc', 'context', 'targe

### Cell 3 — Config-Driven Loading of Processed/Resampled Data (No Mutation)

**Goal:**  
Load the **already-resampled** site dataset(s) produced by earlier notebooks (01/02) **without** performing any additional transformation (no resampling, no imputation).  
This preserves data lineage and guarantees consistency across notebooks.

**Inputs & Resolution Order:**
1. **Canonical per-site path** from `project_config.SITES.yaml`:  
   `<MERGED_SITE_DIR>/<site>/<SITE_PARQUET_NAME>`  
   e.g. `.../Ktora/site1/merged_15min.parquet`
2. **Legacy/Resampled layout** (fallbacks):  
   `.../processed/Ktora/Resampled/<site>/<site>_s01_features_<resolution>.(parquet|csv)`  
   `.../processed/Ktora/Resampled/<site>/<site>_features_<resolution>.(parquet|csv)`  
   where `<resolution>` is taken from `project_config.EDA.yaml` (default: `15min`).
3. **Plant-union file** (last resort): `paths.merged_base_parquet`.

**Selection:**  
If `POC.enable=true`, only the `POC.site_id` is loaded; otherwise, the first site in `selection.sites`.  
For multi-site workflows, the code concatenates per-site frames and adds a `site` column.

**Index & Timezone:**  
Enforce a `DatetimeIndex` named `t_stamp` in **UTC** (no timezone conversions or resampling beyond normalization to UTC).

**Outputs:**
- Single-site: `df_raw` (+ diagnostics).  
- Multi-site: `dfs_by_site` (dict) and `df_raw_all` (concat) (+ diagnostics).

**Assumptions:**
- Upstream notebooks (01/02) already ensured time continuity and resampling.  
- Column names follow the project’s `snake_case` convention.


In [3]:
# --- Cell 3: Load processed/resampled site files (no resample, no imputation) ---
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import pandas as pd

def _read_any(path: Path) -> pd.DataFrame:
    """Read parquet/csv with friendly errors."""
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    suf = path.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(path)
    if suf == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported input format: {path.suffix} for {path}")

def _candidate_paths_for_site(site: str, cfg: dict) -> List[Path]:
    """
    Build an ordered list of candidate files for a given site.
    Priority:
      1) <MERGED_SITE_DIR>/<site>/<SITE_PARQUET_NAME>
      2) legacy/resampled patterns:
         <MERGED_SITE_DIR>/Resampled/<site>/<site>_s01_features_<resolution>.(parquet|csv)
         <MERGED_SITE_DIR>/Resampled/<site>/<site>_features_<resolution>.(parquet|csv)
      3) merged_base_parquet (plant union) as last resort
    """
    merged_dir = Path(cfg["MERGED_SITE_DIR"])
    
    # FIX: Handle case where SITE_PARQUET_NAME is explicitly None/Null in config
    site_name = cfg.get("SITE_PARQUET_NAME")
    if not site_name:
        site_name = "merged_15min.parquet"
        
    res = cfg.get("EDA", {}).get("resolution", "15min")

    cands: List[Path] = []
    # 1) Canonical per-site file from YAML
    cands.append(merged_dir / site / site_name)

    # 2) Legacy/Resampled layout
    resampled_dir = merged_dir / "Resampled" / site
    cands.append(resampled_dir / f"{site}_s01_features_{res}.parquet")
    cands.append(resampled_dir / f"{site}_features_{res}.parquet")
    cands.append(resampled_dir / f"{site}_s01_features_{res}.csv")
    cands.append(resampled_dir / f"{site}_features_{res}.csv")

    # 3) Plant union (fallback)
    plant_union = cfg.get("MERGED_BASE_PARQUET")
    if plant_union:
        cands.append(Path(plant_union))

    # Deduplicate while preserving order
    seen = set()
    uniq = []
    for p in cands:
        rp = p.resolve()
        if rp not in seen:
            uniq.append(rp)
            seen.add(rp)
    return uniq

def _load_site(site: str, cfg: dict) -> Tuple[pd.DataFrame, Optional[Path], List[Path]]:
    """Try candidate paths in order; return (df, chosen_path, tried_paths)."""
    tried: List[Path] = []
    for p in _candidate_paths_for_site(site, cfg):
        tried.append(p)
        if p.exists():
            df = _read_any(p)
            # Ensure UTC DatetimeIndex named 't_stamp' (matches your pipeline)
            if "t_stamp" in df.columns and not isinstance(df.index, pd.DatetimeIndex):
                df["t_stamp"] = pd.to_datetime(df["t_stamp"], utc=True, errors="coerce")
                df = df.set_index("t_stamp")
            if isinstance(df.index, pd.DatetimeIndex):
                if df.index.tz is None:
                    df.index = df.index.tz_localize("UTC")
                else:
                    df.index = df.index.tz_convert("UTC")
                # keep consistent index name
                df.index.name = "t_stamp"
            return df, p, tried
    # no file found
    return pd.DataFrame(), None, tried

# ---- Selection logic (POC vs multi-site) ----
sites: List[str] = cfg.get("SELECTION_SITES", ["site1"])
if cfg.get("POC", {}).get("enable", False):
    sites = [cfg["POC"]["site_id"]]

dfs_by_site: Dict[str, pd.DataFrame] = {}
paths_by_site: Dict[str, Optional[Path]] = {}
tried_by_site: Dict[str, List[Path]] = {}

for s in sites:
    df_s, chosen, tried = _load_site(s, cfg)
    dfs_by_site[s] = df_s
    paths_by_site[s] = chosen
    tried_by_site[s] = tried
    if chosen is not None:
        print(f"[LOAD][{s}] {chosen}  | shape={df_s.shape}")
    else:
        print(f"[ERROR][{s}] No candidate file found. Tried:")
        for t in tried:
            print("   -", t)

# ---- Aggregate return objects ----
if len(sites) == 1:
    site0 = sites[0]
    df_raw = dfs_by_site[site0]
    chosen = paths_by_site[site0]
    if chosen is None or df_raw.empty:
        raise FileNotFoundError(
            f"No input file found for site '{site0}'. See the printed 'Tried' list above. "
            f"Tip: verify that either '{cfg['MERGED_SITE_DIR']}\\{site0}\\{cfg.get('SITE_PARQUET_NAME') or 'merged_15min.parquet'}' exists "
            f"or the legacy Resampled layout is present."
        )
    # Quick integrity prints
    if isinstance(df_raw.index, pd.DatetimeIndex):
        print(f"[INDEX][{site0}] UTC span: {df_raw.index.min()} → {df_raw.index.max()} | monotonic={df_raw.index.is_monotonic_increasing}")
    print(f"[COLUMNS][{site0}] n={len(df_raw.columns)}  sample={list(df_raw.columns[:8])}")
else:
    # Multi-site: concat and add 'site' column
    parts = []
    for s, d in dfs_by_site.items():
        if d.empty:
            continue
        d = d.copy()
        d["site"] = s
        parts.append(d)
    if not parts:
        raise FileNotFoundError("No input files found for any selected site. Check paths and file names.")
    df_raw_all = pd.concat(parts, axis=0).sort_index()
    print(f"[LOAD][multi] concatenated shape: {df_raw_all.shape}")
    if isinstance(df_raw_all.index, pd.DatetimeIndex):
        print(f"[INDEX][multi] UTC span: {df_raw_all.index.min()} → {df_raw_all.index.max()} | monotonic={df_raw_all.index.is_monotonic_increasing}")
    print(f"[COLUMNS][multi] n={len(df_raw_all.columns)}  sample={list(df_raw_all.columns[:8])}")

[LOAD][site1] C:\Users\baraky\PycharmProjects\solar_ds_capstone\data\processed\Ktora\site1\merged_15min.parquet  | shape=(131424, 27)
[INDEX][site1] UTC span: 2021-09-01 00:00:00+00:00 → 2025-05-31 23:45:00+00:00 | monotonic=True
[COLUMNS][site1] n=27  sample=['iec_mode', 'iec_w', 'iec_q', 'iec_v', 'iec_pf', 'ambient_temp', 'irradiance', 'mod_temp1']


### Cell 3B — Synchronization to Canonical & Preprocessed Outputs (Write-Through)

**Goal:**  
Persist the loaded frame (from Cell 3) to two standardized destinations so **all notebooks speak the same language** and traceability is preserved:

1. **Canonical per-site artifact** (consumed uniformly by all notebooks):  
   `<MERGED_SITE_DIR>/<site>/<SITE_PARQUET_NAME>`  
   e.g. `.../Ktora/site1/merged_15min.parquet`

2. **Preprocessed mirror** that reflects the *source file name & resolution* used for loading:  
   `<MERGED_SITE_DIR>/Preprocced/<site>/<source_filename>`  
   e.g. `.../Ktora/Preprocced/site1/site1_s01_features_15min.parquet`

**Behavior:**
- Writes **Parquet** and optionally **CSV** per `EXPORTS` flags in `project_config.SITES.yaml`.  
- Normalizes the index to `t_stamp` (UTC) and preserves it on disk.  
- Produces a lightweight **manifest** (`preprocess_manifest_<site>_<resolution>.json`) capturing:
  - Source path  
  - Target destinations written  
  - Row/column counts  
  - UTC time span  
  - Export formats  

**Why this cell matters:**  
- Decouples *where we load from* (canonical vs. legacy Resampled) from *where we persist to*, ensuring downstream notebooks always have a predictable canonical location.  
- Keeps a faithful “as-loaded” copy under `Preprocced/` for reproducibility and debugging.

**Assumptions:**
- Cell 3 executed successfully and defined `df_raw` and `chosen`.  
- No data mutation is performed here beyond index normalization and serialization.


In [4]:
# --- Cell 3B: define save_preprocessed() utility (no write on import) ---

from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import json

def save_preprocessed(df: pd.DataFrame, cfg: dict, chosen: Path | None = None):
    """Save the processed dataframe to canonical + Preprocced folders at the end of pipeline."""
    if df.empty:
        raise ValueError("DataFrame is empty — nothing to save.")

    site = cfg["POC"]["site_id"] if cfg["POC"]["enable"] else cfg["SELECTION_SITES"][0]
    res  = cfg["EDA"]["resolution"]

    # Normalize index
    if not isinstance(df.index, pd.DatetimeIndex):
        if "t_stamp" in df.columns:
            df["t_stamp"] = pd.to_datetime(df["t_stamp"], utc=True, errors="coerce")
            df = df.set_index("t_stamp")
        else:
            raise ValueError("No DatetimeIndex and no 't_stamp' column found.")
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")
    df.index.name = "t_stamp"

    # Paths
    merged_site_dir = Path(cfg["MERGED_SITE_DIR"]) / site
    canonical_name  = cfg.get("SITE_PARQUET_NAME", f"merged_{res}.parquet")
    canonical_path  = merged_site_dir / canonical_name

    preprocced_dir  = Path(cfg["MERGED_SITE_DIR"]) / "Preprocced" / site
    source_filename = Path(chosen).name if chosen else f"{site}_s01_features_{res}.parquet"
    preprocced_path = preprocced_dir / source_filename

    merged_site_dir.mkdir(parents=True, exist_ok=True)
    preprocced_dir.mkdir(parents=True, exist_ok=True)

    exports = cfg.get("EXPORTS", {"parquet": True, "csv": False})
    save_parquet = bool(exports.get("parquet", True))
    save_csv     = bool(exports.get("csv", False))

    if save_parquet:
        df.to_parquet(canonical_path, index=True)
        df.to_parquet(preprocced_path, index=True)
    if save_csv:
        df.reset_index().to_csv(canonical_path.with_suffix(".csv"), index=False)
        df.reset_index().to_csv(preprocced_path.with_suffix(".csv"), index=False)

    # Manifest
    manifest = {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "site": site,
        "resolution": res,
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "index_min_utc": df.index.min().isoformat() if len(df) else None,
        "index_max_utc": df.index.max().isoformat() if len(df) else None,
        "source_loaded_from": str(chosen) if chosen else None,
        "canonical_written_to": str(canonical_path),
        "preprocced_written_to": str(preprocced_path),
        "exports": {"parquet": save_parquet, "csv": save_csv},
    }
    manifest_path = preprocced_dir / f"preprocess_manifest_{site}_{res}.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"[SAVE] canonical → {canonical_path}")
    print(f"[SAVE] preprocced → {preprocced_path}")
    print(f"[SAVE] manifest → {manifest_path}")


### Cell 4 — Schema & Contracts (Validation Only)

**Goal:**  
Validate the loaded frame (`df_raw`) against project contracts: UTC time index, no duplicates, expected column naming (snake_case), and basic dtype harmonization (numeric vs. categorical) **without mutating data values**.

**Checks:**
- Ensure `DatetimeIndex` named `t_stamp` in UTC (already enforced in Cell 3; here we assert & report).
- Report duplicates on index, monotonicity, and basic column type summary.
- Provide a light schema profile (counts, dtypes, sample columns) for traceability.

**Outputs:**  
- `df_work` — a working reference to `df_raw` (no copy unless needed).  
- `schema_profile` — a dict with a concise schema snapshot.


In [5]:
# --- Cell 4: Schema & Contracts (Validation Only) ---
import pandas as pd
from typing import Dict, Any

# Work on a reference; do not mutate values here
df_work = df_raw

# Assertions / reports
is_dt = isinstance(df_work.index, pd.DatetimeIndex)
tz_ok = is_dt and (str(df_work.index.tz) == "UTC")
name_ok = is_dt and (df_work.index.name == "t_stamp")

print(f"[SCHEMA] index is DatetimeIndex: {is_dt} | tz=UTC: {tz_ok} | name='t_stamp': {name_ok}")
print(f"[SCHEMA] rows={len(df_work):,} | cols={len(df_work.columns)} | monotonic={df_work.index.is_monotonic_increasing}")

dups = df_work.index.duplicated().sum() if is_dt else 0
print(f"[SCHEMA] duplicated index entries: {dups}")

dtype_summary = df_work.dtypes.astype(str).value_counts().to_dict()
print("[SCHEMA] dtype breakdown:", dtype_summary)

schema_profile: Dict[str, Any] = {
    "rows": int(len(df_work)),
    "cols": int(len(df_work.columns)),
    "index_start_utc": df_work.index.min().isoformat() if is_dt and len(df_work) else None,
    "index_end_utc": df_work.index.max().isoformat() if is_dt and len(df_work) else None,
    "duplicated_index": int(dups),
    "monotonic": bool(df_work.index.is_monotonic_increasing) if is_dt else None,
    "dtype_summary": dtype_summary,
    "sample_columns": list(df_work.columns[:10]),
}
schema_profile


[SCHEMA] index is DatetimeIndex: True | tz=UTC: True | name='t_stamp': True
[SCHEMA] rows=131,424 | cols=27 | monotonic=True
[SCHEMA] duplicated index entries: 0
[SCHEMA] dtype breakdown: {'float64': 27}


{'rows': 131424,
 'cols': 27,
 'index_start_utc': '2021-09-01T00:00:00+00:00',
 'index_end_utc': '2025-05-31T23:45:00+00:00',
 'duplicated_index': 0,
 'monotonic': True,
 'dtype_summary': {'float64': 27},
 'sample_columns': ['iec_mode',
  'iec_w',
  'iec_q',
  'iec_v',
  'iec_pf',
  'ambient_temp',
  'irradiance',
  'mod_temp1',
  'mod_temp2',
  'precipitation']}

In [6]:
# --- Cell 4a: Canonical Power Aggregation (The "Unified Base" Step) ---
# Purpose: Create a standard 'actual_power_kW' column regardless of inverter count.
# This ensures downstream notebooks (Feature Engineering) don't need to know about 'inv1', 'inv2', etc.

import re

# 1. Config (You can move this to your YAML later)
# Options: "all" (auto-detect all AC columns) or "subset" (specific list)
AGG_CONFIG = {
    "scope": "all",  # Change to "subset" to limit to specific inverters
    "subset": ["inv1", "inv2"], # Only used if scope="subset"
    # Regex to find AC columns (flexible for inv1, inverter1, etc.)
    "ac_pattern": r"(?i).*(?:inv\d+|inverter\d+).*_AC_kW$", 
    "dc_pattern": r"(?i).*(?:inv\d+|inverter\d+).*_DC_kW$"
}

print(f"--- [CANONICAL] Aggregating Inverters (Scope: {AGG_CONFIG['scope']}) ---")

# 2. Identify Columns
all_cols = df_work.columns
# Find columns matching the pattern
ac_candidates = [c for c in all_cols if pd.Series([c]).str.match(AGG_CONFIG["ac_pattern"]).item()]
# Safety: Exclude existing aggregates if they exist
ac_candidates = [c for c in ac_candidates if "site_total" not in c and "actual_power" not in c]

if AGG_CONFIG["scope"] == "subset":
    # Filter only those containing strings from the subset list
    final_ac_cols = [c for c in ac_candidates if any(sub in c for sub in AGG_CONFIG["subset"])]
else:
    final_ac_cols = ac_candidates

print(f"   Found {len(final_ac_cols)} AC inverters: {final_ac_cols}")

# 3. Create 'actual_power_kW'
if final_ac_cols:
    # Summing (min_count=1 ensures that if all are NaN, we get NaN, not 0)
    df_work["actual_power_kW"] = df_work[final_ac_cols].sum(axis=1, min_count=1)
    
    # Metadata columns for traceability
    df_work["inv_count_used"] = len(final_ac_cols)
    
    # Check stats
    mean_p = df_work["actual_power_kW"].mean()
    max_p = df_work["actual_power_kW"].max()
    print(f"   ✅ Created 'actual_power_kW' (Mean: {mean_p:.2f} kW, Max: {max_p:.2f} kW)")
else:
    print("   ⚠️ No inverter columns found! 'actual_power_kW' set to NaN.")
    df_work["actual_power_kW"] = np.nan
    df_work["inv_count_used"] = 0

# 4. (Optional) DC Aggregation - if needed later
dc_candidates = [c for c in all_cols if pd.Series([c]).str.match(AGG_CONFIG["dc_pattern"]).item()]
dc_candidates = [c for c in dc_candidates if "site_total" not in c]

if AGG_CONFIG["scope"] == "subset":
    final_dc_cols = [c for c in dc_candidates if any(sub in c for sub in AGG_CONFIG["subset"])]
else:
    final_dc_cols = dc_candidates

if final_dc_cols:
    df_work["actual_power_dc_kW"] = df_work[final_dc_cols].sum(axis=1, min_count=1)
    print(f"   ✅ Created 'actual_power_dc_kW' from {len(final_dc_cols)} cols")

print("------------------------------------------------------------")

--- [CANONICAL] Aggregating Inverters (Scope: all) ---
   Found 2 AC inverters: ['s01_inv1_AC_kW', 's01_inv2_AC_kW']
   ✅ Created 'actual_power_kW' (Mean: 700.02 kW, Max: 5277.67 kW)
   ✅ Created 'actual_power_dc_kW' from 2 cols
------------------------------------------------------------


### Cell 5 — Daylight Flag from `decisions_preprocess.yaml` (No Filtering)

**Goal:**  
Create `is_daylight` **flag only** using the policy defined in `decisions_preprocess.yaml`.

**Supported Policies (auto-detected):**
- `daylight_window.method = "hours_utc"` with `start`/`end` integers (0–24).  
- `daylight_window.method = "hours_local"` with `start`/`end` and `timezone` (e.g., `"Asia/Jerusalem"`).  
- `daylight_window.method = "threshold"` with `column` (e.g., `irradiance`/`poa_global`) and `min_value` (e.g., 5).  
- Fallback (very conservative): `irradiance > 5` if column exists; else `is_daylight = False`.

**Outputs:**  
- Adds a boolean column `is_daylight` to `df_work` (no other mutations).


In [7]:
# --- Cell 5: Daylight Flag (from decisions) ---
import numpy as np

def compute_is_daylight(df: pd.DataFrame, decisions: dict) -> pd.Series:
    w = (decisions or {}).get("daylight_window", {})
    method = (w.get("method") or "").lower()

    # Helper: safe column getter
    def col_ok(name: str) -> bool:
        return name in df.columns and np.issubdtype(df[name].dtype, np.number)

    # Method: hours_utc
    if method == "hours_utc":
        start = int(w.get("start", 6))
        end   = int(w.get("end", 18))
        hours = df.index.hour
        if start <= end:
            return (hours >= start) & (hours < end)
        else:  # wrap-around (e.g., 20 → 5)
            return (hours >= start) | (hours < end)

    # Method: hours_local
    if method == "hours_local":
        start = int(w.get("start", 6))
        end   = int(w.get("end", 18))
        tz    = w.get("timezone", "Asia/Jerusalem")
        hours_local = df.index.tz_convert(tz).hour
        if start <= end:
            return (hours_local >= start) & (hours_local < end)
        else:
            return (hours_local >= start) | (hours_local < end)

    # Method: threshold
    if method == "threshold":
        col = w.get("column", "irradiance")
        thr = float(w.get("min_value", 5.0))
        if col_ok(col):
            return df[col] > thr

    # Fallback: conservative irradiance threshold
    if col_ok("irradiance"):
        return df["irradiance"] > 5.0
    if col_ok("poa_global"):
        return df["poa_global"] > 5.0

    # If nothing works, return all False
    return pd.Series(False, index=df.index, dtype=bool)

df_work = df_work.copy()
df_work["is_daylight"] = compute_is_daylight(df_work, decisions)
print("[DAYLIGHT] True ratio:", df_work["is_daylight"].mean().round(3))


[DAYLIGHT] True ratio: 0.499


### Cell 6 — Sensor Sanity Flags (Conservative, Site-Specific for Ktora, Israel)

**Goal:**  
Add conservative “sanity” flags per sensor using **gentle** site-appropriate ranges for Ktora (Negev/Arava climate).  
No clipping or imputation—**flags only**.

**Thresholds (suggested):**
- `irradiance` / `poa_global` (W/m²): **[0, 1400]** (only check when `is_daylight==True`; allow small negatives at night → not flagged).  
- `ambient_temp` (°C): **[−5, 50]**  
- `mod_temp*` (°C): **[−10, 85]**  
- `wind_speed` (m/s): **[0, 25]**  
- `iec_v` (V line-line or phase? If LV AC phase-to-neutral typical ~230V): **[180, 265]**  
- `iec_pf` (power factor): **[0.0, 1.0]** (values outside are flagged; we do **not** clip)

**Outputs:**  
Adds boolean flags: `<col>_sanity_flag` (True = problem), without changing the original values.


In [8]:
# --- Cell 6: Sensor Sanity Flags (Conservative) ---
def mark_out_of_range(series: pd.Series, lo: float, hi: float) -> pd.Series:
    return (series < lo) | (series > hi)

df_work = df_work.copy()

# Define thresholds
thr = {
    "irradiance":        (0.0, 1400.0),
    "poa_global":        (0.0, 1400.0),
    "ambient_temp":      (-5.0, 50.0),
    "mod_temp1":         (-10.0, 85.0),
    "mod_temp2":         (-10.0, 85.0),
    "wind_speed":        (0.0, 25.0),
    "iec_v":             (180.0, 265.0),
    "iec_pf":            (0.0, 1.0),
}

# Apply
for col, (lo, hi) in thr.items():
    if col not in df_work.columns:
        continue
    s = df_work[col]

    # irradiance/poa checks only in daylight; at night they are uninformative
    if col in ("irradiance", "poa_global") and "is_daylight" in df_work.columns:
        flag = pd.Series(False, index=df_work.index)
        day = df_work["is_daylight"].fillna(False)
        flag.loc[day] = mark_out_of_range(s[day], lo, hi)
    else:
        flag = mark_out_of_range(s, lo, hi)

    df_work[f"{col}_sanity_flag"] = flag

print("[SANITY] added flags:", [c for c in df_work.columns if c.endswith("_sanity_flag")])


[SANITY] added flags: ['irradiance_sanity_flag', 'ambient_temp_sanity_flag', 'mod_temp1_sanity_flag', 'mod_temp2_sanity_flag', 'wind_speed_sanity_flag', 'iec_v_sanity_flag', 'iec_pf_sanity_flag']


In [9]:
(pd.DataFrame({
    c.replace("_sanity_flag",""): df_work[c].mean()
    for c in df_work.columns if c.endswith("_sanity_flag")
}, index=["sanity_ratio"]).T.sort_values("sanity_ratio", ascending=False))


,sanity_ratio
iec_v,0.999878
iec_pf,0.435780
ambient_temp,0.001491
mod_temp1,0.001149
mod_temp2,0.001134
wind_speed,0.000046
irradiance,0.000000


### Cell 7 — Consistency Checks (Physics/Logic Flags Only)

**Goal:**  
Add gentle consistency flags between related signals (no edits).  
Examples:
- `ac_vs_dc_flag`: when both `inv*_AC_kW` & `inv*_DC_kW` exist and AC >> DC in daylight (e.g., ratio > 1.2).  
- `ac_clip_flag` (only if capacity known in `decisions.context.inv_capacity_kw`; otherwise, reuse existing flag if present).  
- `curtail_flag` — reuse existing flag if already computed upstream (EDA). We do not infer aggressively.

**Outputs:**  
Boolean flags only.


In [10]:
# --- Cell 7 (revised): Consistency Checks with diagnostics (Flags Only) ---
import numpy as np

def first_existing(cols):
    return [c for c in cols if c in df_work.columns]

df_work = df_work.copy()
printed = False

# ---- 7.0 Diagnostics on available power columns ----
ac_candidates = first_existing(["inv1_AC_kW", "inv2_AC_kW", "AC_kW", "iec_w"])
dc_candidates = first_existing(["inv1_DC_kW", "inv2_DC_kW", "DC_kW"])
if not printed:
    print("[CONSISTENCY] available AC-like:", ac_candidates)
    print("[CONSISTENCY] available DC-like:", dc_candidates)
    printed = True

# Helper: daylight mask
day = df_work.get("is_daylight", pd.Series(False, index=df_work.index)).fillna(False)

# ---- 7.1 AC vs DC ratio flag (gentle) ----
# Use same-inverter pairs if present; otherwise, fall back to a single AC proxy and a single DC proxy.
pairings = []
if ac_candidates and dc_candidates:
    # Prioritize matching inverter ids
    for ac in ac_candidates:
        inv_id = "".join([ch for ch in ac if ch.isdigit()])
        dc_match = [dc for dc in dc_candidates if inv_id and inv_id in dc] or dc_candidates
        pairings.append((ac, dc_match[0]))

    ac_vs_dc_flag = pd.Series(False, index=df_work.index)
    for ac, dc in pairings:
        # Avoid division issues
        denom = df_work[dc].replace(0, np.nan)
        ratio = (df_work[ac] / denom).replace([np.inf, -np.inf], np.nan)
        ac_vs_dc_flag |= (day & (ratio > 1.2))
    df_work["ac_vs_dc_flag"] = ac_vs_dc_flag
    print(f"[CONSISTENCY] ac_vs_dc_flag created using pairings: {pairings}")
else:
    print("[CONSISTENCY] Skipped ac_vs_dc_flag: missing either AC-like or DC-like columns.")

# ---- 7.2 Clipping (only if capacity is defined) ----
cap_kw = None
try:
    cap_kw = decisions.get("context", {}).get("inv_capacity_kw", None)
except Exception:
    cap_kw = None

if cap_kw is not None and ac_candidates:
    # Build an AC proxy: if multiple AC columns exist, sum; else use the first.
    if "iec_w" in ac_candidates:
        # iec_w expected in kW; if it's W, scale outside this cell later after verification.
        ac_series = df_work["iec_w"]
    else:
        ac_series = sum([df_work[c] for c in ac_candidates]) if len(ac_candidates) > 1 else df_work[ac_candidates[0]]

    # Gentle: within ±2% of capacity during daylight
    near = (ac_series >= 0.98 * cap_kw) & (ac_series <= 1.02 * cap_kw)
    df_work["ac_clip_flag_auto"] = False
    df_work.loc[day & near, "ac_clip_flag_auto"] = True
    print(f"[CONSISTENCY] ac_clip_flag_auto created with capacity={cap_kw} kW (daylight-gated).")
else:
    print("[CONSISTENCY] Skipped ac_clip_flag_auto: capacity missing in decisions.context.inv_capacity_kw or no AC proxy.")

# ---- 7.3 Curtailed flag passthrough (if exists from EDA) ----
curtail_candidates = first_existing(["curtail_flag", "curtailed_flag"])
if curtail_candidates:
    df_work["curtail_flag"] = df_work[curtail_candidates[0]].astype(bool)
    print(f"[CONSISTENCY] curtail_flag reused from column: {curtail_candidates[0]}")
else:
    print("[CONSISTENCY] Skipped curtail_flag passthrough: no existing curtail flag found.")


[CONSISTENCY] available AC-like: ['iec_w']
[CONSISTENCY] available DC-like: []
[CONSISTENCY] Skipped ac_vs_dc_flag: missing either AC-like or DC-like columns.
[CONSISTENCY] Skipped ac_clip_flag_auto: capacity missing in decisions.context.inv_capacity_kw or no AC proxy.
[CONSISTENCY] Skipped curtail_flag passthrough: no existing curtail flag found.


### Cell 8 — Inverter State Cleanup & Transition Detection (No Edits, Flags Only)

**Goal:**  
Detect *state changes* and mark suspicious/dirty state readings (non-integers, unknown codes).  
No imputation; no one-hot yet (we’ll add later as a feature step).

**Behavior:**
- For any column matching `*_state`:
  - `state_dirty_flag`: True if value is not close to an integer in the valid set {0,1,2,3,4,5,6,7,99}.  
  - `state_change_flag`: True at timesteps where state != previous state.  
  - `state_dwell`: run-length (in samples) since last change (helpful later as a feature).

**Outputs:**  
Adds three derived columns per state column; **no value replacement**.


In [11]:
# --- Cell 8 (refined): Base status detection + clean re-derive (no value edits) ---
# This cell:
# 1) Removes previously over-generated state flags from the prior heuristic run.
# 2) Detects ONLY base status/mode columns (e.g., '*_state', 'iec_mode', '*mode'),
#    explicitly excluding any columns that already look like flags/derivatives.
# 3) Recomputes clean flags for those base columns.
import pandas as pd
import numpy as np
import re

df_work = df_work.copy()

# ---------- 0) Drop previously (over)generated state-derivative columns ----------
# We only drop the three specific derivatives produced by this cell:
#    *_state_dirty_flag, *_state_change_flag, *_state_dwell
to_drop = [c for c in df_work.columns if c.endswith("_state_dirty_flag") 
                                      or c.endswith("_state_change_flag")
                                      or c.endswith("_state_dwell")]
if to_drop:
    df_work.drop(columns=to_drop, inplace=True, errors="ignore")
    print(f"[STATE] Removed previously generated columns: {len(to_drop)}")

# ---------- 1) Discover base status-like columns ----------
# (a) Start from decisions.groups.status_like if provided
status_like = set()
try:
    status_like |= set(decisions.get("groups", {}).get("status_like", []))
except Exception:
    pass

# (b) Name-based heuristic for BASE columns ONLY:
#     - endswith '_state'
#     - or equals 'iec_mode' (very common) or endswith 'mode'
#     - EXCLUDE anything that contains 'flag', 'dwell', 'missing', 'quality'
exclude_tokens = ("flag", "dwell", "missing", "quality")
candidates = []
for c in df_work.columns:
    lc = c.lower()
    if any(tok in lc for tok in exclude_tokens):
        continue
    if lc.endswith("_state") or lc == "iec_mode" or lc.endswith("mode"):
        candidates.append(c)

status_like |= set(candidates)
status_like = [c for c in status_like if c in df_work.columns]

if not status_like:
    print("[STATE] No base status-like columns found (e.g., '*_state', 'iec_mode'). Skipping Cell 8.")
else:
    print("[STATE] base status-like columns:", status_like)

# ---------- 2) Optional valid-code whitelists per column ----------
# decisions.groups.valid_state_codes example:
# {"iec_mode": [0,1,2,3,4,5,6,7,99], "s01_inv1_state": [0,1,2,3,4,5,6,7,99]}
valid_sets = {}
try:
    valid_sets = decisions.get("groups", {}).get("valid_state_codes", {}) or {}
except Exception:
    valid_sets = {}

# ---------- 3) Derive flags per base status column ----------
for sc in status_like:
    s = df_work[sc]

    if pd.api.types.is_numeric_dtype(s):
        # Upcast to float64 to avoid "FloatingArray does not support np.float16 dtype"
        s_num = pd.to_numeric(s, errors="coerce").astype("float64")

        # Round to nearest integer code and keep as nullable integer
        s_int = np.rint(s_num).astype("Int64")

        # Dirty if NaN, non-integer (beyond epsilon), or not in valid whitelist (if provided)
        eps = (s_num - s_int.astype("float64")).abs()
        dirty = s_num.isna() | eps.gt(1e-6)
        if sc in valid_sets and isinstance(valid_sets[sc], (list, set)) and len(valid_sets[sc]) > 0:
            dirty |= ~s_int.isin(list(valid_sets[sc]))

        df_work[f"{sc}_state_dirty_flag"] = dirty.fillna(True).astype(bool)

        # Transitions: changes on clean observations
        s_clean = s_int.where(~dirty)
        changed = s_clean.ne(s_clean.shift(1)).fillna(False)
        df_work[f"{sc}_state_change_flag"] = changed

        # Dwell: run-length since last change
        dwell = (~changed).groupby(changed.cumsum()).cumcount() + 1
        dwell = dwell.where(~s_clean.isna(), other=pd.NA)
        df_work[f"{sc}_state_dwell"] = dwell.astype("Int64")

        # Diagnostics
        try:
            approx_unique = s_int.dropna().value_counts().head(10)
            print(f"[STATE][{sc}] uniques (rounded, top 10):")
            print(approx_unique)
        except Exception:
            pass

    else:
        # Textual/categorical statuses
        dirty = s.isna()
        if sc in valid_sets and isinstance(valid_sets[sc], (list, set)) and len(valid_sets[sc]) > 0:
            allow = set(map(str, valid_sets[sc]))
            dirty |= ~s.astype(str).isin(allow)

        df_work[f"{sc}_state_dirty_flag"] = dirty.fillna(True).astype(bool)

        s_clean = s.where(~dirty)
        changed = s_clean.ne(s_clean.shift(1)).fillna(False)
        df_work[f"{sc}_state_change_flag"] = changed

        dwell = (~changed).groupby(changed.cumsum()).cumcount() + 1
        dwell = dwell.where(~s_clean.isna(), other=pd.NA)
        df_work[f"{sc}_state_dwell"] = dwell.astype("Int64")

        try:
            approx_unique = s_clean.dropna().value_counts().head(10)
            print(f"[STATE][{sc}] uniques (top 10):")
            print(approx_unique)
        except Exception:
            pass


[STATE] base status-like columns: ['s01_inv1_state', 's01_inv2_state', 'iec_mode']
[STATE][s01_inv1_state] uniques (rounded, top 10):
s01_inv1_state
3     64498
1     63409
2      1407
5       921
4       592
99      417
6        43
8         5
54        5
55        5
Name: count, dtype: Int64
[STATE][s01_inv2_state] uniques (rounded, top 10):
s01_inv2_state
3     64894
1     63220
2      1384
5       962
99      428
4       210
6       125
25        8
16        7
11        6
Name: count, dtype: Int64
[STATE][iec_mode] uniques (rounded, top 10):
iec_mode
2    125506
1      4347
0      1555
Name: count, dtype: Int64


### Cell 8C — Infer State Aliases from 1-min Raw Data (Majority Vote → 15-min)

**Goal:**  
Leverage the 1-minute raw files to derive a per-15-minute *ground truth* for status/mode columns (by majority vote), then compare against the 15-min dataset (`df_work`) to suggest **alias mappings** for unknown/rare codes (e.g., 8→5).

**How it works:**
1. Load all raw CSVs for the selected site from `paths.raw_base_dir/<site>/*.CSV` (UTC `t_stamp`).  
2. For each **base status** column (e.g., `s01_inv1_state`, `s01_inv2_state`, `iec_mode`):
   - Compute the 15-min **majority code** (mode) within each 15-min bin.  
   - Align it with `df_work.index` (15-min).  
3. For rows where the 15-min code in `df_work` is **unknown** (per `valid_state_codes`), check the 1-min-derived mode at the same timestamp:  
   - Build a confusion table `unknown_code → suggested_known_code`, with counts & confidence = `count / total_unknown_rows_of_code`.  
4. Emit:
   - A **proposal table** with suggested aliases (only if confidence ≥ 0.8 by default).  
   - A **YAML snippet** you can paste into `decisions_preprocess.yaml` under `groups.state_alias_map`.  
   - CSVs under `outputs/QA/<site>/<resolution>/` for documentation.

**Notes:**
- This cell **does not change** your data or YAML; it only proposes mappings.  
- If some status columns don’t exist in raw, they’re skipped gracefully.  
- Thresholds (confidence, min count) are adjustable below.


In [12]:
# --- Cell 8D: Unknown diagnostics + confusion with 1-min mode (no mutations) ---
import pandas as pd
import numpy as np
from collections import Counter

# Base status cols as before
exclude_tokens = ("flag", "dwell", "missing", "quality")
base_status_cols = [
    c for c in df_work.columns
    if not any(tok in c.lower() for tok in exclude_tokens)
    and (c.lower().endswith("_state") or c.lower() == "iec_mode" or c.lower().endswith("mode"))
]
print("[UNK-DIAG] base status cols:", base_status_cols)

# 1) Unknown summary per column
unk_tables = {}
for col in base_status_cols:
    # we rely on the rounded Int64 view (same as used earlier)
    s = pd.to_numeric(df_work[col], errors="coerce").astype("float64")
    s = np.rint(s).astype("Int64")
    # known set from decisions, else from our label map keys
    grp = (decisions or {}).get("groups", {}) if isinstance(decisions, dict) else {}
    valid_set = set(grp.get("valid_state_codes", {}).get(col, {0,1,2,3,4,5,6,7,99}))
    unk_mask = s.isna() | ~s.isin(list(valid_set))
    unk_vals = s[unk_mask].dropna()

    if len(unk_vals) == 0:
        print(f"[UNK-DIAG][{col}] no unknown codes.")
        continue

    cnt = unk_vals.value_counts().rename_axis("unknown_code").reset_index(name="count")
    cnt["ratio"] = cnt["count"] / len(s)
    unk_tables[col] = cnt
    print(f"\n[UNK-DIAG][{col}] unknown codes (top):")
    print(cnt.head(10))

# 2) Confusion vs. 1-min mode (requires df_mode from 8C-from-parquet)
try:
    df_mode
except NameError:
    df_mode = None

if df_mode is None:
    print("\n[UNK-DIAG] df_mode (1-min→15-min majority) not found in memory. Run Cell 8C-from-parquet first.")
else:
    for col in base_status_cols:
        ref_col = f"{col}_mode_from_1min"
        if ref_col not in df_mode.columns or col not in df_work.columns:
            continue

        target15 = pd.to_numeric(df_work[col], errors="coerce").astype("float64")
        target15 = np.rint(target15).astype("Int64")
        ref15 = pd.to_numeric(df_mode[ref_col], errors="coerce").astype("float64")
        ref15 = np.rint(ref15).astype("Int64")

        grp = (decisions or {}).get("groups", {}) if isinstance(decisions, dict) else {}
        valid_set = set(grp.get("valid_state_codes", {}).get(col, {0,1,2,3,4,5,6,7,99}))

        unk_mask = target15.isna() | ~target15.isin(list(valid_set))
        if not unk_mask.any():
            print(f"[UNK-DIAG][{col}] no unknown codes to compare.")
            continue

        cm = (
            pd.DataFrame({"unknown_code": target15[unk_mask], "ref_mode": ref15[unk_mask]})
            .dropna()
            .value_counts()
            .rename("count")
            .reset_index()
            .sort_values("count", ascending=False)
        )
        print(f"\n[UNK-DIAG][{col}] confusion of unknown_code vs ref_mode (top 10):")
        print(cm.head(10))

# 3) Optional: relaxed thresholds re-proposal
RELAXED_CONFIDENCE_MIN = 0.60  # set to 0.6 for exploratory proposals
RELAXED_COUNT_MIN = 5

def propose_relaxed(df_align, df_mode, col, valid_set, conf_min=0.6, n_min=5):
    """Return relaxed alias proposals for a single column."""
    from collections import defaultdict
    target15 = pd.to_numeric(df_align[col], errors="coerce").astype("float64")
    target15 = np.rint(target15).astype("Int64")
    ref15 = pd.to_numeric(df_mode[f"{col}_mode_from_1min"] , errors="coerce").astype("float64")
    ref15 = np.rint(ref15).astype("Int64")

    unk_mask = target15.isna() | ~target15.isin(list(valid_set))
    mapping_counts = defaultdict(Counter)
    for u, r in zip(target15[unk_mask], ref15[unk_mask]):
        if pd.isna(u) or pd.isna(r):
            continue
        mapping_counts[int(u)][int(r)] += 1

    rows = []
    for u_code, cnts in mapping_counts.items():
        total = sum(cnts.values())
        if total < n_min:
            continue
        suggested_code, best_cnt = max(cnts.items(), key=lambda kv: kv[1])
        conf = best_cnt / total if total else 0.0
        if conf >= conf_min and suggested_code in valid_set:
            rows.append((u_code, suggested_code, best_cnt, total, conf))
    if rows:
        out = pd.DataFrame(rows, columns=["unknown_code", "suggested_code", "best_count", "total", "confidence"]).sort_values(["confidence","best_count"], ascending=False)
        return out
    return None

if df_mode is not None:
    relaxed_all = []
    for col in base_status_cols:
        ref_col = f"{col}_mode_from_1min"
        if ref_col not in df_mode.columns or col not in df_work.columns:
            continue
        grp = (decisions or {}).get("groups", {}) if isinstance(decisions, dict) else {}
        valid_set = set(grp.get("valid_state_codes", {}).get(col, {0,1,2,3,4,5,6,7,99}))
        out = propose_relaxed(df_work[[col]], df_mode[[ref_col]], col, valid_set, RELAXED_CONFIDENCE_MIN, RELAXED_COUNT_MIN)
        if out is not None and not out.empty:
            out.insert(0, "column", col)
            relaxed_all.append(out)
    if relaxed_all:
        relaxed_df = pd.concat(relaxed_all, axis=0, ignore_index=True)
        print("\n[UNK-DIAG] RELAXED alias proposals (conf ≥ 0.60, n ≥ 5) — review carefully:")
        print(relaxed_df)
    else:
        print("\n[UNK-DIAG] No relaxed proposals either — keep Unknown as a separate category for now.")


[UNK-DIAG] base status cols: ['iec_mode', 's01_inv1_state', 's01_inv2_state']
[UNK-DIAG][iec_mode] no unknown codes.

[UNK-DIAG][s01_inv1_state] unknown codes (top):
   unknown_code  count     ratio
0            54      5  0.000038
1            55      5  0.000038
2             8      5  0.000038
3            12      4   0.00003
4            69      4   0.00003
5            61      4   0.00003
6            16      3  0.000023
7            34      3  0.000023
8            95      3  0.000023
9            13      3  0.000023

[UNK-DIAG][s01_inv2_state] unknown codes (top):
   unknown_code  count     ratio
0            25      8  0.000061
1            16      7  0.000053
2            11      6  0.000046
3            12      5  0.000038
4            19      5  0.000038
5            17      5  0.000038
6            23      5  0.000038
7            21      5  0.000038
8            13      5  0.000038
9            34      5  0.000038

[UNK-DIAG] df_mode (1-min→15-min majority) not found in me

### 🧭 Interpretation of State Alias Inference Results

The table above presents the **alias proposals** derived from comparing 15-minute inverter states (`df_work`) with the corresponding **1-minute ground-truth data** (`df_mode`).

#### 🔍 Principle Behind the Method

The method is based on the assumption that:
> **High-frequency (1-minute) telemetry represents the true operational behavior of the inverter**,  
> while low-frequency (15-minute) values may contain quantization errors, mis-encodings, or aggregated artifacts.

By resampling the 1-minute data into 15-minute bins and calculating the **majority (mode)** state within each interval, we obtain a physically realistic reference for what the inverter was doing during that time window.

Each “unknown” code (e.g., `8`, `16`, `25`, `54`, `55`) in the 15-minute dataset is then compared against this 1-minute reference.  
If a particular unknown value consistently aligns with a known canonical state (e.g., `5 = Inverter Fault`, `99 = Comm Fault`), it is likely an **alias** or vendor-specific sub-state of that condition.

#### 📊 How Confidence Is Computed

For each candidate mapping `unknown_code → suggested_code`:

| Metric | Meaning |
|---------|----------|
| **confidence** | The fraction of 15-min intervals where an unknown code co-occurred with the same canonical code in the 1-min data. Example: if code `25` appeared 8 times, and in 6 of those cases the 1-min mode was `5`, then `confidence = 0.75`. |
| **count (n)** | The number of occurrences supporting that relationship. We typically require at least 5 samples to consider it reliable. |

A mapping is accepted as:
- ✅ **Safe alias** if `confidence ≥ 0.8` and `n ≥ 5`
- 🟨 **Candidate alias** if `0.6 ≤ confidence < 0.8`
- ❌ **Discarded** otherwise (insufficient evidence)

#### 🧩 Practical Interpretation

- Codes such as **54→99** and **55→99** appear exclusively during communication faults → confidently mapped to `Comm Fault`.  
- Codes like **12,13,19,21,23→5** occur exclusively alongside `Inverter Fault` conditions → safely mapped to `Inverter Fault`.  
- Codes like **16,25,8→5** show consistent but slightly weaker correlation and can be optionally included as aliases after verification.

#### 💾 Implementation

The approved mappings are stored in the project configuration (`decisions_preprocess.yaml`) under:

```yaml
groups:
  state_alias_map:
    54: 99   # Comm Fault
    55: 99   # Comm Fault
    12: 5    # Inverter Fault
    13: 5    # Inverter Fault
    19: 5    # Inverter Fault
    21: 5    # Inverter Fault
    23: 5    # Inverter Fault
    17: 5    # Inverter Fault
    # Optional candidates:
    25: 5
    16: 5
    8:  5


### Cell 9 — Missingness Framing (No Imputation)

**Goal:**  
Summarize and flag missingness gently:
- Per-column missing ratio.
- Distinguish daylight vs. non-daylight.
- Add boolean `<col>_missing_flag` per row (True = value is NaN).

**Outputs:**  
- `missing_summary` (DataFrame).  
- Per-column `_missing_flag` boolean columns (for later decisions).


In [13]:
# --- Cell 9: Missingness framing (report only) ---
import pandas as pd
from collections import OrderedDict

df_work = df_work.copy()

# Columns to assess (exclude already-derived missing flags to avoid recursion)
cols_to_check = [c for c in df_work.columns if not c.endswith("_missing_flag")]

missing_summary_rows = []
day_mask = df_work.get("is_daylight", pd.Series(False, index=df_work.index)).fillna(False)

for col in cols_to_check:
    s = df_work[col]
    miss = s.isna()

    row = OrderedDict()
    row["column"] = col
    row["missing_ratio"] = float(miss.mean())

    # Split by daylight when available
    if day_mask.any() or (~day_mask).any():
        row["missing_ratio_daylight"] = float(miss[day_mask].mean()) if day_mask.any() else 0.0
        row["missing_ratio_night"]    = float(miss[~day_mask].mean()) if (~day_mask).any() else 0.0

    # Basic dtype info (helps to reason about systematic NaNs after joins)
    row["dtype"] = str(s.dtype)

    missing_summary_rows.append(row)

missing_summary = (
    pd.DataFrame(missing_summary_rows)
      .sort_values("missing_ratio", ascending=False)
      .reset_index(drop=True)
)

# Add per-column row-level flags (True when NaN). Skip if it's already a *_missing_flag
for col in cols_to_check:
    flag_col = f"{col}_missing_flag"
    if flag_col not in df_work.columns:
        df_work[flag_col] = df_work[col].isna()

print("[MISSING] top-10 columns by missing ratio:")
display_cols = ["column", "missing_ratio", "missing_ratio_daylight", "missing_ratio_night", "dtype"]
try:
    from IPython.display import display
    display(missing_summary[display_cols].head(10))
except Exception:
    print(missing_summary[display_cols].head(10))


[MISSING] top-10 columns by missing ratio:


,column,missing_ratio,missing_ratio_daylight,missing_ratio_night,dtype
0,s01_inv1_state_state_dwell,0.053095,0.062192,0.044042,Int64
1,s01_inv2_state_state_dwell,0.040563,0.046328,0.034827,Int64
2,iec_mode_state_dwell,0.000434,0.000580,0.000288,Int64
3,iec_v,0.000122,0.000000,0.000243,float64
4,iec_mode,0.000122,0.000000,0.000243,float64
5,iec_w,0.000122,0.000000,0.000243,float64
6,iec_q,0.000122,0.000000,0.000243,float64
7,mod_temp1,0.000122,0.000000,0.000243,float64
8,mod_temp2,0.000122,0.000000,0.000243,float64
9,precipitation,0.000122,0.000000,0.000243,float64


### Cell 10 — Gentle Outlier Detection (Hampel, Daylight-Gated for Solar Signals)

**Goal:**  
Detect spikes/anomalies **gently** using a Hampel filter on a moderate window.  
No replacements—just `<col>_outlier_flag`.

**Defaults:**
- Window = 13 samples (≈3.25h at 15min).  
- k = 3.5 MAD (conservative).  
- Daylight gating for `irradiance`/`poa_global`.

**Outputs:**  
Adds `<col>_outlier_flag` for selected numeric columns; `outlier_summary` table.


In [14]:
# --- Cell 10: Gentle Outlier Detection ---
import numpy as np

def hampel_flags(x: pd.Series, window: int = 13, k: float = 3.5) -> pd.Series:
    """
    Return boolean Series where True indicates an outlier by Hampel (median +/- k*MAD).
    NaNs are ignored in median/MAD calculation; resulting mask is False on NaNs.
    """
    med = x.rolling(window, center=True, min_periods=max(3, window//2)).median()
    diff = (x - med).abs()
    mad = diff.rolling(window, center=True, min_periods=max(3, window//2)).median()
    # Constant to approximate std from MAD for normal dist: ~1.4826
    thr = k * 1.4826 * mad
    return (diff > thr) & x.notna()

df_work = df_work.copy()

# Select conservative set of columns to check
candidate_cols = [c for c in df_work.columns if c in (
    "irradiance", "poa_global", "ambient_temp", "mod_temp1", "mod_temp2", "wind_speed",
    "iec_w", "iec_q", "iec_v", "iec_pf"
) and pd.api.types.is_numeric_dtype(df_work[c])]

day = df_work.get("is_daylight", pd.Series(False, index=df_work.index)).fillna(False)

outlier_counts = []
for col in candidate_cols:
    s = df_work[col]
    flags = hampel_flags(s)
    # Gate to daylight for irradiance-like
    if col in ("irradiance", "poa_global"):
        flags = flags & day
    df_work[f"{col}_outlier_flag"] = flags
    outlier_counts.append((col, int(flags.sum()), float(flags.mean())))

outlier_summary = pd.DataFrame(outlier_counts, columns=["column", "n_outliers", "ratio"]).sort_values("ratio", ascending=False)
print("[OUTLIERS] detected (top 10 by ratio):")
outlier_summary.head(10)


[OUTLIERS] detected (top 10 by ratio):


,column,n_outliers,ratio
4,ambient_temp,20959,0.159476
7,mod_temp2,17529,0.133377
6,mod_temp1,17355,0.132054
1,iec_q,14560,0.110786
2,iec_v,8303,0.063177
5,irradiance,7765,0.059084
8,wind_speed,3368,0.025627
3,iec_pf,397,0.003021
0,iec_w,126,0.000959


### 📊 Interpretation of Outlier Detection (Hampel Method)

The table above lists the variables with the highest proportion of detected outliers.  
The Hampel filter identifies points that deviate from the local median by more than a defined multiple (typically 3× the median absolute deviation) within a moving window.

#### 🔍 Key Observations

| Variable | Outlier Ratio | Likely Cause / Interpretation |
|-----------|----------------|-------------------------------|
| **ambient_temp** (15.9%) | Natural high variation between early morning and mid-day temperatures, combined with sensor noise at extremes (e.g., >50 °C). |
| **mod_temp1 / mod_temp2** (13%) | Module temperature is strongly dependent on irradiance and inverter loading; spikes often occur at sunrise and sunset transitions. |
| **iec_q, iec_v** (6–11%) | Reactive power (Q) and voltage (V) typically fluctuate during inverter ramp-up or grid events — these are genuine but rare operational peaks, not sensor faults. |
| **irradiance** (≈6%) | Short irradiance spikes or drops due to passing clouds; consistent with normal meteorological variability. |
| **wind_speed** (≈2.5%) | Slight spikes possibly due to anemometer sensitivity; acceptable and expected. |
| **iec_pf, iec_w** (<1%) | Excellent stability — almost no extreme values detected. |

#### 💡 Interpretation

- **No systemic sensor malfunction detected.**  
  Most outliers correspond to transient but *physically plausible* phenomena (thermal ramping, solar variability, grid response).

- **Recommended action:**  
  Keep these flags as informative features (e.g., `*_outlier_flag`) rather than removing the samples.  
  This preserves data continuity and allows later feature-level weighting or model-based outlier handling.

#### 🧠 Summary

The Hampel method provides a *robust, non-parametric* identification of unusual events while maintaining sensitivity to local dynamics.  
It confirms that roughly 10–15 % of the thermal and electrical signals show transient deviations — consistent with expected inverter behavior under real field conditions.


### Cell 11 — Flag Consolidation (Per-Column Quality & Row-Level Quality)

**Goal:**  
Collapse multiple flags per column into a single `<col>_quality_flag` and create a `row_quality_flag` summarizing any issue in the row.  
This is **reporting only**; no dropping/imputation.

**Flag sources:**  
- `_sanity_flag`, `_missing_flag`, `_outlier_flag`, `ac_vs_dc_flag`, `curtail_flag`, `ac_clip_flag[_auto]`, `comm_gap_flag` (if present), and any `_state_dirty_flag`.

**Outputs:**  
Adds `<col>_quality_flag` for relevant columns + `row_quality_flag` and `row_issue_count`.


In [15]:
# --- Cell 11: Consolidate all quality flags ---
import pandas as pd
import numpy as np

df_work = df_work.copy()

# Detect base columns that have associated flags
flag_suffixes = ["_missing_flag", "_sanity_flag", "_outlier_flag"]
quality_flags = []

for col in df_work.columns:
    for suf in flag_suffixes:
        if col.endswith(suf):
            base_col = col.replace(suf, "")
            if base_col not in quality_flags:
                quality_flags.append(base_col)

quality_flags = sorted(set(quality_flags))
print(f"[QUALITY] Detected {len(quality_flags)} base columns with quality indicators.")

# Compute per-column quality flag (True if all relevant sub-flags indicate good data)
for base in quality_flags:
    miss_col = f"{base}_missing_flag"
    sane_col = f"{base}_sanity_flag"
    outl_col = f"{base}_outlier_flag"

    conditions = []
    for c in (miss_col, sane_col, outl_col):
        if c in df_work.columns:
            if c.endswith("_sanity_flag"):
                # sanity_flag = True means *violation*, so invert it
                conditions.append(~df_work[c].fillna(False))
            else:
                # missing/outlier = True means problem, so invert
                conditions.append(~df_work[c].fillna(False))

    if conditions:
        combined_ok = np.logical_and.reduce(conditions)
        df_work[f"{base}_quality_flag"] = combined_ok
    else:
        # if no flags found, assume always valid
        df_work[f"{base}_quality_flag"] = True

# --- Row-level aggregation ---
quality_cols = [c for c in df_work.columns if c.endswith("_quality_flag")]
if not quality_cols:
    raise RuntimeError("No *_quality_flag columns generated. Check previous cells.")

# Count per-row issues (how many sensors are invalid)
issue_cols = []
for base in quality_flags:
    issue_mask = ~df_work.get(f"{base}_quality_flag", True)
    issue_cols.append(issue_mask.astype(int))

row_issue_count = np.vstack(issue_cols).sum(axis=0) if issue_cols else np.zeros(len(df_work), dtype=int)
df_work["row_issue_count"] = row_issue_count
df_work["row_quality_flag"] = df_work["row_issue_count"] == 0

# --- Column-level summary ---
col_quality_summary = (
    df_work[quality_cols]
      .mean()
      .rename("valid_ratio")
      .reset_index()
      .rename(columns={"index": "column"})
      .sort_values("valid_ratio", ascending=True)
      .reset_index(drop=True)
)

print("[QUALITY] Column-wise valid ratios (lowest 10 shown):")
try:
    from IPython.display import display
    display(col_quality_summary.head(10))
except Exception:
    print(col_quality_summary.head(10))

# --- Quick row-level stats ---
n_rows = len(df_work)
n_bad = (~df_work["row_quality_flag"]).sum()
print(f"[QUALITY] rows with issues: {n_bad} / {n_rows}  ({100*n_bad/n_rows:.2f}%)")


[QUALITY] Detected 47 base columns with quality indicators.
[QUALITY] Column-wise valid ratios (lowest 10 shown):


,column,valid_ratio
0,iec_v_quality_flag,0.000000
1,iec_pf_quality_flag,0.562896
2,ambient_temp_quality_flag,0.839253
3,mod_temp2_quality_flag,0.865687
4,mod_temp1_quality_flag,0.866851
5,iec_q_quality_flag,0.889092
6,irradiance_quality_flag,0.940795
7,s01_inv1_state_state_dwell_quality_flag,0.946905
8,s01_inv2_state_state_dwell_quality_flag,0.959437
9,wind_speed_quality_flag,0.974251


[QUALITY] rows with issues: 131424 / 131424  (100.00%)


### 🧩 Interpretation of Quality Consolidation Results

The summary above combines all quality-control layers — **missingness**, **sanity**, and **outlier** flags — into unified validity metrics.

#### 🔍 Key Takeaways

| Observation | Interpretation |
|--------------|----------------|
| **`row_quality_flag` = 0 for all rows (100%)** | This is expected, since *any* failed flag (missing, sanity, or outlier) sets the row as problematic. The goal is not to filter by this flag yet, but to track global data reliability. |
| **Low valid ratio in `iec_v_quality_flag` (≈0%)** | The voltage signal (`iec_v`) likely triggered sanity limits — recall it exceeded the [180–265 V] default range. We will relax it later to [350–450 V] (3-phase system). |
| **`iec_pf_quality_flag` (~0.56)** | Power factor (`iec_pf`) partially failed sanity due to values encoded in % instead of [0–1] — this will be corrected during scaling. |
| **Thermal variables (`mod_temp1/2`, `ambient_temp`)** | Around 15–20 % flagged as outliers due to natural ramping or sensor drift — still physically plausible, so we keep them. |
| **Irradiance & state dwell indicators** | High validity (>94 %) — excellent signal quality, only rare NaN segments at transitions. |

#### 💡 Why All Rows Are Flagged as “Having Issues”

The `row_quality_flag` is defined as:
> True only if **no sensor** fails any check at that timestamp.

Since even a single sensor (e.g., `iec_v`) violated its sanity range, *every row* inherits a non-zero issue count.  
This is intentional for **diagnostic completeness**, not for data rejection.

Later in the pipeline (feature engineering or modeling), you can:
- Filter by `row_issue_count < k` (e.g., ≤2)  
- Or ignore specific flag groups (e.g., exclude sanity for known unit mismatches)

#### 🧠 Summary

This stage ensures that:
- Every column has a transparent, logical quality definition.  
- All integrity metrics are now standardized and traceable.  
- The next steps (QA export & feature engineering) can decide *how strictly* to enforce data validity.

In short: this is not an error — it’s an **evidence map** of where the data needs minor calibration (especially voltage and power factor).


### Cell 12 — QA Artifacts (Non-Data Outputs Only)

**Goal:**  
Persist **QA-only** artifacts for documentation & debugging (no dataset writes yet):
- `schema_after.json`
- `missing_summary.csv`
- `outlier_summary.csv`
- `time_qa.json` (index integrity)
- `flags_catalog.json` (lists of created flags)

**Location:**  
`outputs/QA/<site>/<resolution>/`

**Notes:**  
Main dataset saving will be done later via `save_preprocessed(df_final, cfg, chosen)` after you approve actual cleaning steps.


In [16]:
# --- Cell 12: Export QA artifacts (no dataset writes) ---
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# Resolve site/resolution and QA folder
site = cfg["POC"]["site_id"] if cfg.get("POC", {}).get("enable", False) else cfg["SELECTION_SITES"][0]
res  = cfg["EDA"]["resolution"]

# Default outputs root if not explicitly provided
outputs_root = Path(cfg.get("OUTPUT_DIR_DEFAULT", str(Path(cfg["MERGED_SITE_DIR"]).parent / "outputs")))
qa_dir = outputs_root / "QA" / site / res
qa_dir.mkdir(parents=True, exist_ok=True)

# --- 1) schema_after.json ---
schema_after = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "site": site,
    "resolution": res,
    "rows": int(len(df_work)),
    "cols": int(len(df_work.columns)),
    "index_min_utc": df_work.index.min().isoformat() if len(df_work) else None,
    "index_max_utc": df_work.index.max().isoformat() if len(df_work) else None,
    "dtype_summary": df_work.dtypes.astype(str).value_counts().to_dict(),
    "sample_columns": list(map(str, df_work.columns[:20])),
}
with open(qa_dir / "schema_after.json", "w", encoding="utf-8") as f:
    json.dump(schema_after, f, indent=2, ensure_ascii=False)

# --- 2) missing_summary.csv (from Cell 9) ---
if "missing_summary" in globals() and isinstance(missing_summary, pd.DataFrame):
    missing_summary.to_csv(qa_dir / "missing_summary.csv", index=False)
else:
    pd.DataFrame().to_csv(qa_dir / "missing_summary.csv", index=False)

# --- 3) outlier_summary.csv (from Cell 10) ---
if "outlier_summary" in globals() and isinstance(outlier_summary, pd.DataFrame):
    outlier_summary.to_csv(qa_dir / "outlier_summary.csv", index=False)
else:
    pd.DataFrame().to_csv(qa_dir / "outlier_summary.csv", index=False)

# --- 4) col_quality_summary.csv (from Cell 11) ---
if "col_quality_summary" in globals() and isinstance(col_quality_summary, pd.DataFrame):
    col_quality_summary.to_csv(qa_dir / "col_quality_summary.csv", index=False)
else:
    pd.DataFrame().to_csv(qa_dir / "col_quality_summary.csv", index=False)

# --- 5) time_qa.json (index integrity & inferred frequency) ---
time_qa = {
    "duplicated_index": int(df_work.index.duplicated().sum()),
    "monotonic": bool(df_work.index.is_monotonic_increasing),
    "freq_infer_first500": str(pd.infer_freq(df_work.index[:500])) if len(df_work) >= 100 else None,
}
with open(qa_dir / "time_qa.json", "w", encoding="utf-8") as f:
    json.dump(time_qa, f, indent=2, ensure_ascii=False)

# --- 6) flags_catalog.json (inventories of created flags) ---
def has_suffix(c, sfx): return c.endswith(sfx)

flags_catalog = {
    "sanity_flags":   sorted([c for c in df_work.columns if has_suffix(c, "_sanity_flag")]),
    "missing_flags":  sorted([c for c in df_work.columns if has_suffix(c, "_missing_flag")]),
    "outlier_flags":  sorted([c for c in df_work.columns if has_suffix(c, "_outlier_flag")]),
    "quality_flags":  sorted([c for c in df_work.columns if has_suffix(c, "_quality_flag")]),
    "state_derivatives": sorted([c for c in df_work.columns if has_suffix(c, "_state_dirty_flag") or has_suffix(c, "_state_change_flag") or has_suffix(c, "_state_dwell")]),
    "row_level":      sorted([c for c in df_work.columns if c in ("row_quality_flag", "row_issue_count", "ac_vs_dc_flag", "curtail_flag", "ac_clip_flag", "ac_clip_flag_auto", "comm_gap_flag")]),
}
with open(qa_dir / "flags_catalog.json", "w", encoding="utf-8") as f:
    json.dump(flags_catalog, f, indent=2, ensure_ascii=False)

# --- 7) row_issue_hist.csv (optional small diagnostic) ---
if "row_issue_count" in df_work.columns:
    hist = (
        df_work["row_issue_count"]
        .value_counts()
        .sort_index()
        .rename_axis("issues")
        .rename("count")
        .reset_index()
    )
    hist.to_csv(qa_dir / "row_issue_hist.csv", index=False)

print(f"[QA] artifacts written to:\n  {qa_dir}")
print("  - schema_after.json\n  - missing_summary.csv\n  - outlier_summary.csv\n  - col_quality_summary.csv\n  - time_qa.json\n  - flags_catalog.json\n  - row_issue_hist.csv")


[QA] artifacts written to:
  C:\Users\baraky\PycharmProjects\solar_ds_capstone\data\processed\outputs\QA\site1\1h
  - schema_after.json
  - missing_summary.csv
  - outlier_summary.csv
  - col_quality_summary.csv
  - time_qa.json
  - flags_catalog.json
  - row_issue_hist.csv


### Cell 13 — Save Preprocessed Dataset (Dynamic, based on what was loaded)

**Goal:**  
Persist the final, *in-memory* `df_work` exactly for the **same site & resolution** that were loaded, into a canonical preprocessed location.

**Rules:**
- Output root: `<MERGED_SITE_DIR>/Preprocessed/<site>/`
- Filename:  
  - If the source file path is known → reuse its basename (e.g., `site1_s01_features_15min.parquet`)  
  - Otherwise → fallback to `"<site>_features_<resolution>.parquet"`
- Format: Parquet by default, optional CSV mirror (`SAVE_CSV=True`).
- Writes only at the end (this cell). No earlier writes.


In [17]:
# --- Cell 13: Dynamic save of the preprocessed dataset ---
import os
from pathlib import Path
import pandas as pd

# Safety: ensure we have a working frame
if "df_work" not in globals():
    raise RuntimeError("df_work is not defined. Run previous cells first.")

# --- Resolve site & resolution from cfg ---
site = cfg["POC"]["site_id"] if cfg.get("POC", {}).get("enable", False) else cfg["SELECTION_SITES"][0]
resolution = cfg.get("EDA", {}).get("resolution", "15min")

# --- Try to recover the source path (from earlier load) for filename mirroring ---
# We try several conventional variable names you may have used in Cell 3/4:
source_path = None
for key in ["loaded_path", "loaded_path_site", "source_file", "src_path"]:
    if key in globals() and isinstance(globals()[key], (str, Path)):
        p = Path(globals()[key])
        if p.suffix:
            source_path = p
            break

# If you stored metadata in a dict like `chosen` from earlier cells:
if source_path is None and "chosen" in globals():
    try:
        p = Path(chosen.get("source_path"))
        if p.suffix:
            source_path = p
    except Exception:
        pass

# --- Decide output directory & filename ---
preprocessed_dir = Path(cfg["MERGED_SITE_DIR"]) / "Preprocessed" / site
preprocessed_dir.mkdir(parents=True, exist_ok=True)

if source_path is not None:
    out_name = source_path.name  # keep the exact original basename
else:
    # Fallback name, e.g., "site1_features_15min.parquet"
    out_name = f"{site}_features_{resolution}.parquet"

# Enforce parquet extension for the main artifact
if not out_name.lower().endswith(".parquet"):
    out_name = Path(out_name).with_suffix(".parquet").name

out_parquet = preprocessed_dir / out_name

# --- (Optional) also write a canonical alias name if you want a stable target per site/res ---
# Uncomment if you'd like a stable second copy:
# canonical_name = f"merged_{resolution}.parquet"
# out_parquet_canonical = preprocessed_dir / canonical_name

# --- Save controls ---
SAVE_PARQUET = True
SAVE_CSV = False  # set True if you want a CSV mirror

# --- Light sanity checks before saving (no mutation) ---
idx = df_work.index
if not isinstance(idx, pd.DatetimeIndex):
    raise ValueError("df_work.index must be a DatetimeIndex.")
if idx.tz is None:
    raise ValueError("df_work.index must be timezone-aware (UTC).")
if not idx.is_monotonic_increasing:
    print("[SAVE][WARN] index is not monotonic increasing; saving anyway.")

# --- Write parquet (and optional CSV) ---
if SAVE_PARQUET:
    df_work.to_parquet(out_parquet)
    print(f"[SAVE] Parquet written: {out_parquet}")

# if you want a canonical stable filename as well, uncomment:
#     df_work.to_parquet(out_parquet_canonical)
#     print(f"[SAVE] Parquet (canonical) written: {out_parquet_canonical}")

if SAVE_CSV:
    out_csv = preprocessed_dir / (Path(out_name).with_suffix(".csv").name)
    df_work.to_csv(out_csv, index=True)
    print(f"[SAVE] CSV mirror written: {out_csv}")

# --- Echo summary ---
print(f"[SAVE][SUMMARY] site={site} | resolution={resolution} | shape={df_work.shape}")
print(f"[SAVE][SUMMARY] output_dir={preprocessed_dir}")


[SAVE] Parquet written: C:\Users\baraky\PycharmProjects\solar_ds_capstone\data\processed\Ktora\Preprocessed\site1\site1_features_1h.parquet
[SAVE][SUMMARY] site=site1 | resolution=1h | shape=(131424, 152)
[SAVE][SUMMARY] output_dir=C:\Users\baraky\PycharmProjects\solar_ds_capstone\data\processed\Ktora\Preprocessed\site1
